In [1]:
!pip install -q transformers accelerate bitsandbytes gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
SYSTEM_PROMPT = """Yor are a Medibot, a helpful medical infromation asssistant.

RULES YOU MUST FOLLOW STRICTLY:
1. Only answer questions related to medicine, health, symptoms, diseases,
   treatments, medications, anatomy, or wellness.
2. If the user asks anything NOT related to medical/health topics
   (e.g. coding, math, general chit-chat, entertainment, politics),
   respond ONLY with: "Sorry, I can answer only medical-related queries."
3. Do not answer non-medical questions even if the user insists, jokes,
   or tries to rephrase the request as medical.
4. Always remind users to consult a real doctor for serious concerns.
5. Keep answers clear, concise, and non-alarming.
"""

In [7]:
FEWSHOT = """
Example 1:
User: What causes a migraine?
Assistant: Migraines are often caused by triggers like stress, hormonal changes, certain foods, or lack of sleep...

Example 2:
User: Can you write me a Python script?
Assistant: Sorry, I can answer only medical-related queries.

Example 3:
User: What's the capital of France?
Assistant: Sorry, I can answer only medical-related queries.
"""

In [8]:
def build_prompt(history,user_input):
  messages = [{"role":"system","content":"SYSTEM_PROMPT + FEWSHOT"}]
  for u,a in history:
    messages.append({"role":"user","content": u})
    messages.append({"role":"assistant","content": a})
  messages.append({"role":"user","content": user_input})
  return messages

In [9]:
def generate_response(history,user_input):
  messages = build_prompt(history,user_input)
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(text, return_tensors="pt").to(model.device)

  output = model.generate(
      **inputs,
      max_new_tokens=300,
      temperature=0.4,
      do_sample=True,
      pad_token_id=tokenizer.eos_token_id
  )
  response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
  return response.strip()

In [10]:
MEDICAL_KEYWORDS = ["symptom", "disease", "pain", "medicine", "doctor", "health",
                     "treatment", "diagnosis", "fever", "infection", "medication", "body"]

def is_likely_medical(text):
    return any(kw in text.lower() for kw in MEDICAL_KEYWORDS)

In [11]:
import gradio as gr

chat_history = []

def chat(user_input, chat_history_ui):
  response = generate_response(chat_history, user_input)
  chat_history.append((user_input, response))
  chat_history_ui.append((user_input, response))
  return "", chat_history_ui

with gr.Blocks() as demo:
  chatbot = gr.Chatbot(label="Medibot")
  msg = gr.Textbox(placeholder="Ask a medical question...")
  msg.submit(chat, [msg, chatbot], [msg, chatbot])

demo.launch(share= True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de8ae8044e22f46f4a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
